In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell 08.1 - Notebook overview and project setup
# Define the feasibility question, project paths, and stopping rule.

from pathlib import Path
from IPython.display import display, Markdown
PROJECT_ROOT = _repo_root()
NOTEBOOK_DIR = PROJECT_ROOT / "03_Notebooks" / "04_Genome_Comparison"
INTERMEDIATE_DIR = PROJECT_ROOT / "04_Intermediate" / "08_Whole_Chromosomal_Sequence"
RESULTS_TABLE_DIR = PROJECT_ROOT / "05_Results" / "Tables"
OLD_PROJECT_ROOT = _previous_project_root(PROJECT_ROOT)

assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"
assert NOTEBOOK_DIR.exists(), f"Notebook folder not found: {NOTEBOOK_DIR}"
assert OLD_PROJECT_ROOT.exists(), f"Previous project not found: {OLD_PROJECT_ROOT}"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_TABLE_DIR.mkdir(parents=True, exist_ok=True)

overview = """
# Notebook 08 — Whole-Chromosomal-Sequence Comparison Feasibility

## Objective

Determine whether the **176 blaTEM-1-only E. coli genomes** can support an
unbiased whole-chromosomal-sequence comparison.

This notebook is a **go/no-go feasibility notebook only**. It does not test
sequence variation against ceftazidime MIC.

The feasibility assessment asks whether:

1. the 176 pathogens have a verified one-to-one assembly mapping;
2. all 176 genome assemblies can be obtained and read correctly;
3. assembly quality is adequate for whole-sequence comparison;
4. chromosome/plasmid separation can be handled before discovery;
5. the total sequence scale is computationally manageable.

No genes, loci, SNPs, or sequence features are preselected.

Variable sequence elements will be defined only after this feasibility stage
supports proceeding.
"""

display(Markdown(overview))

print("Notebook folder:", NOTEBOOK_DIR)
print("Intermediate folder:", INTERMEDIATE_DIR)
print("Results table folder:", RESULTS_TABLE_DIR)

print(
    "\nTransition: Cell 08.2 will verify the established one-to-one "
    "BioSample-to-assembly mapping."
)

In [ ]:
#@title Cell 08.2 - Verify the established 176-pathogen assembly mapping
# Use the authoritative mapping table already identified in this project.

import pandas as pd
from pathlib import Path
from IPython.display import display

MAPPING_PATH = (
    PROJECT_ROOT
    / "05_Results"
    / "Tables"
    / "01_blaTEM-1_only_176_pathogens.csv"
)

assert MAPPING_PATH.exists(), f"Mapping file not found: {MAPPING_PATH}"

mapping = pd.read_csv(MAPPING_PATH, low_memory=False)

required = {"biosample", "assembly_accession"}
missing = required - set(mapping.columns)

assert not missing, f"Required mapping columns missing: {sorted(missing)}"

mapping = mapping.copy()

mapping["biosample"] = (
    mapping["biosample"]
    .astype(str)
    .str.extract(r"(SAMN\d+)", expand=False)
    .str.upper()
)

mapping["assembly_accession"] = (
    mapping["assembly_accession"]
    .astype(str)
    .str.extract(r"(GC[AF]_\d+(?:\.\d+)?)", expand=False)
    .str.upper()
)

mapping = mapping.loc[
    mapping["biosample"].notna()
    & mapping["assembly_accession"].notna()
].copy()

per_biosample = (
    mapping.groupby("biosample")["assembly_accession"]
    .nunique()
)

print("Mapping verification:")
print("Unique BioSamples:", mapping["biosample"].nunique())
print("BioSamples with one assembly accession:", int((per_biosample == 1).sum()))
print("BioSamples with multiple assembly accessions:", int((per_biosample > 1).sum()))
print("Unique assembly accessions:", mapping["assembly_accession"].nunique())

assert mapping["biosample"].nunique() == 176, (
    f"Expected 176 BioSamples; found {mapping['biosample'].nunique()}."
)

assert (per_biosample == 1).all(), (
    "At least one BioSample has more than one assembly accession."
)

mapping_176 = (
    mapping[["biosample", "assembly_accession"]]
    .drop_duplicates()
    .sort_values("biosample")
    .reset_index(drop=True)
)

assert len(mapping_176) == 176, (
    f"Expected 176 one-to-one mappings; found {len(mapping_176)}."
)

print("\nFirst five verified mappings:")
display(mapping_176.head())

print(
    "\nTransition: Cell 08.3 will attach ceftazidime MIC and freeze "
    "the authoritative 176-pathogen manifest."
)

In [ ]:
#@title Cell 08.3 - Freeze the authoritative 176-pathogen manifest
# Attach MIC to the verified BioSample-to-assembly mapping and save one fixed manifest.

import pandas as pd
from pathlib import Path
from IPython.display import display

COHORT_PATH = (
    OLD_PROJECT_ROOT
    / "05_Association_Analysis"
    / "Notebook10"
    / "10_beta_lactamase_only_group_membership.csv"
)

OUTPUT_PATH = (
    RESULTS_TABLE_DIR
    / "08_authoritative_176_assembly_manifest.csv"
)

assert COHORT_PATH.exists(), f"Cohort file not found: {COHORT_PATH}"

cohort = pd.read_csv(COHORT_PATH, low_memory=False)

required_cohort = {"biosample", "log2_mic"}
missing = required_cohort - set(cohort.columns)

assert not missing, f"Cohort columns missing: {sorted(missing)}"

cohort = cohort[["biosample", "log2_mic"]].copy()

cohort["biosample"] = (
    cohort["biosample"]
    .astype(str)
    .str.extract(r"(SAMN\d+)", expand=False)
    .str.upper()
)

cohort["log2_mic"] = pd.to_numeric(
    cohort["log2_mic"],
    errors="coerce"
)

cohort = cohort.loc[
    cohort["biosample"].isin(set(mapping_176["biosample"]))
].copy()

mic_counts = (
    cohort.dropna(subset=["log2_mic"])
    .groupby("biosample")["log2_mic"]
    .nunique()
)

assert len(mic_counts) == 176, (
    f"MIC values were found for {len(mic_counts)} of 176 BioSamples."
)

assert (mic_counts == 1).all(), (
    "At least one BioSample has conflicting log2 MIC values."
)

mic = (
    cohort.dropna(subset=["log2_mic"])
    .drop_duplicates(["biosample", "log2_mic"])
    .drop_duplicates("biosample")
    .reset_index(drop=True)
)

manifest = mapping_176.merge(
    mic,
    on="biosample",
    how="left",
    validate="one_to_one"
)

assert len(manifest) == 176
assert manifest["biosample"].nunique() == 176
assert manifest["assembly_accession"].nunique() == 176
assert manifest["log2_mic"].notna().all()

manifest = manifest.sort_values("biosample").reset_index(drop=True)
manifest.to_csv(OUTPUT_PATH, index=False)

print("Authoritative 176-pathogen manifest verified.\n")
print("Pathogens:", len(manifest))
print("Unique BioSamples:", manifest["biosample"].nunique())
print("Unique assembly accessions:", manifest["assembly_accession"].nunique())
print("MIC values present:", manifest["log2_mic"].notna().sum())

print("\nMIC summary:")
display(manifest["log2_mic"].describe())

print("\nFirst five records:")
display(manifest.head())

print("\nSaved:", OUTPUT_PATH)

print(
    "\nTransition: Cell 08.4 will obtain the exact 176 assemblies "
    "listed in this frozen manifest."
)

In [ ]:
#@title Cell 08.4 - Obtain the exact 176 genome assemblies
# Download only the assembly accessions in the frozen manifest and verify one genome FASTA per accession.

import shutil
import subprocess
import zipfile
from pathlib import Path
import pandas as pd
from IPython.display import display

MANIFEST_PATH = (
    RESULTS_TABLE_DIR
    / "08_authoritative_176_assembly_manifest.csv"
)

DOWNLOAD_ROOT = (
    INTERMEDIATE_DIR
    / "NCBI_assemblies_176"
)

ZIP_PATH = (
    INTERMEDIATE_DIR
    / "NCBI_assemblies_176.zip"
)

ACCESSION_LIST_PATH = (
    INTERMEDIATE_DIR
    / "08_assembly_accessions_176.txt"
)

OUTPUT_PATH = (
    RESULTS_TABLE_DIR
    / "08_downloaded_176_assembly_manifest.csv"
)

manifest = pd.read_csv(MANIFEST_PATH)

assert len(manifest) == 176
assert manifest["assembly_accession"].nunique() == 176

datasets_exe = shutil.which("datasets")

assert datasets_exe is not None, (
    "NCBI datasets CLI is not available in this runtime. "
    "Stop here and install/restore the same NCBI datasets CLI used previously "
    "before continuing."
)

print("NCBI datasets CLI:")
subprocess.run(
    [datasets_exe, "version"],
    check=True
)

with open(ACCESSION_LIST_PATH, "w") as fh:
    for accession in manifest["assembly_accession"]:
        fh.write(str(accession) + "\n")

print("\nAssembly accession list saved:")
print(ACCESSION_LIST_PATH)

data_root = DOWNLOAD_ROOT / "ncbi_dataset" / "data"

if not data_root.exists():

    print("\nDownloading the 176 exact assembly accessions...")

    subprocess.run(
        [
            datasets_exe,
            "download",
            "genome",
            "accession",
            "--inputfile",
            str(ACCESSION_LIST_PATH),
            "--include",
            "genome",
            "--filename",
            str(ZIP_PATH),
            "--no-progressbar",
        ],
        check=True
    )

    assert ZIP_PATH.exists(), "NCBI download archive was not created."

    DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DOWNLOAD_ROOT)

else:
    print("\nExisting extracted NCBI dataset found; no new download performed.")


def genomic_fasta_candidates(accession):
    accession_dir = data_root / str(accession)

    if not accession_dir.exists():
        return []

    candidates = []

    for p in accession_dir.rglob("*.fna"):
        name = p.name.lower()

        if any(
            token in name
            for token in [
                "_cds_from_genomic",
                "_rna_from_genomic",
            ]
        ):
            continue

        candidates.append(p)

    return sorted(candidates)


rows = []

for r in manifest.itertuples(index=False):

    candidates = genomic_fasta_candidates(
        r.assembly_accession
    )

    rows.append(
        {
            "biosample": r.biosample,
            "assembly_accession": r.assembly_accession,
            "log2_mic": r.log2_mic,
            "n_genome_fasta_candidates": len(candidates),
            "assembly_fasta_path": (
                str(candidates[0])
                if len(candidates) == 1
                else None
            ),
        }
    )

download_manifest = pd.DataFrame(rows)

print("\nGenome FASTA resolution:")
display(
    download_manifest["n_genome_fasta_candidates"]
    .value_counts()
    .sort_index()
    .rename_axis("n_FASTA_files")
    .reset_index(name="n_assemblies")
)

problem = download_manifest.loc[
    download_manifest["n_genome_fasta_candidates"] != 1
]

if not problem.empty:
    print("\nAssemblies not resolving to exactly one genome FASTA:")
    display(problem)

download_manifest.to_csv(OUTPUT_PATH, index=False)

print("\nSaved:", OUTPUT_PATH)

assert problem.empty, (
    "At least one assembly did not resolve to exactly one genome FASTA. "
    "Review the output before continuing."
)

print(
    "\nTransition: Cell 08.5 will verify the downloaded sequence files "
    "and summarize assembly quality."
)

In [ ]:
#@title Cell 08.5 - Verify sequence integrity and assembly quality
# Measure genome size, contig count, N50, GC fraction, and ambiguous sequence.

from pathlib import Path
import math
import pandas as pd
from IPython.display import display

INPUT_PATH = (
    RESULTS_TABLE_DIR
    / "08_downloaded_176_assembly_manifest.csv"
)

OUTPUT_PATH = (
    RESULTS_TABLE_DIR
    / "08_assembly_sequence_QC.csv"
)

download_manifest = pd.read_csv(INPUT_PATH)

assert len(download_manifest) == 176
assert download_manifest["assembly_fasta_path"].notna().all()


def fasta_metrics(path):
    lengths = []
    total_gc = 0
    total_atgc = 0
    ambiguous = 0
    current_len = 0

    with open(path, "rt", errors="ignore") as fh:
        for line in fh:

            if line.startswith(">"):
                if current_len > 0:
                    lengths.append(current_len)
                current_len = 0
                continue

            seq = line.strip().upper()

            if not seq:
                continue

            current_len += len(seq)

            a = seq.count("A")
            c = seq.count("C")
            g = seq.count("G")
            t = seq.count("T")

            atgc = a + c + g + t

            total_gc += g + c
            total_atgc += atgc
            ambiguous += len(seq) - atgc

    if current_len > 0:
        lengths.append(current_len)

    total_bp = sum(lengths)

    if not lengths or total_bp == 0:
        return None

    half = total_bp / 2
    running = 0
    n50 = 0

    for length in sorted(lengths, reverse=True):
        running += length
        if running >= half:
            n50 = length
            break

    return {
        "total_bp": total_bp,
        "n_contigs": len(lengths),
        "largest_contig_bp": max(lengths),
        "N50_bp": n50,
        "GC_fraction": (
            total_gc / total_atgc
            if total_atgc > 0
            else float("nan")
        ),
        "ambiguous_bp": ambiguous,
        "ambiguous_fraction": (
            ambiguous / total_bp
            if total_bp > 0
            else float("nan")
        ),
    }


rows = []

for r in download_manifest.itertuples(index=False):

    path = Path(r.assembly_fasta_path)

    if not path.exists():
        metrics = None
    else:
        metrics = fasta_metrics(path)

    if metrics is None:
        rows.append(
            {
                "biosample": r.biosample,
                "assembly_accession": r.assembly_accession,
                "sequence_readable": False,
            }
        )
        continue

    rows.append(
        {
            "biosample": r.biosample,
            "assembly_accession": r.assembly_accession,
            "sequence_readable": True,
            **metrics,
        }
    )

qc = pd.DataFrame(rows)

qc["critical_size_flag"] = (
    (qc["total_bp"] < 3_000_000)
    | (qc["total_bp"] > 8_000_000)
)

qc["critical_ambiguity_flag"] = (
    qc["ambiguous_fraction"] > 0.10
)

qc["critical_sequence_flag"] = (
    (~qc["sequence_readable"])
    | qc["critical_size_flag"].fillna(True)
    | qc["critical_ambiguity_flag"].fillna(True)
)

qc.to_csv(OUTPUT_PATH, index=False)

print("Sequence-file verification:")
print("Readable assemblies:", int(qc["sequence_readable"].sum()), "/ 176")
print("Critical sequence-QC flags:", int(qc["critical_sequence_flag"].sum()))

print("\nAssembly summary:")
display(
    qc[
        [
            "total_bp",
            "n_contigs",
            "largest_contig_bp",
            "N50_bp",
            "GC_fraction",
            "ambiguous_fraction",
        ]
    ].describe().T
)

if qc["critical_sequence_flag"].any():
    print("\nAssemblies with critical QC flags:")
    display(
        qc.loc[
            qc["critical_sequence_flag"],
            [
                "biosample",
                "assembly_accession",
                "total_bp",
                "n_contigs",
                "N50_bp",
                "GC_fraction",
                "ambiguous_fraction",
                "critical_size_flag",
                "critical_ambiguity_flag",
            ],
        ]
    )

print("\nSaved:", OUTPUT_PATH)

print(
    "\nTransition: Cell 08.6 will determine how much chromosome/plasmid "
    "separation is available directly from the assembly FASTA records."
)

In [ ]:
#@title Cell 08.6 - Assess direct chromosome/plasmid separation
# Inspect FASTA record labels without yet removing or classifying any sequence.

from pathlib import Path
import pandas as pd
from IPython.display import display

INPUT_PATH = (
    RESULTS_TABLE_DIR
    / "08_downloaded_176_assembly_manifest.csv"
)

OUTPUT_PATH = (
    RESULTS_TABLE_DIR
    / "08_chromosome_plasmid_label_QC.csv"
)

download_manifest = pd.read_csv(INPUT_PATH)


def record_lengths_by_label(path):
    records = []
    header = None
    seq_len = 0

    def classify(text):
        lower = text.lower()

        if "plasmid" in lower:
            return "plasmid-labelled"

        if "chromosome" in lower:
            return "chromosome-labelled"

        return "unlabelled"

    with open(path, "rt", errors="ignore") as fh:
        for line in fh:

            if line.startswith(">"):

                if header is not None:
                    records.append(
                        (
                            classify(header),
                            seq_len,
                        )
                    )

                header = line[1:].strip()
                seq_len = 0

            else:
                seq_len += len(line.strip())

    if header is not None:
        records.append(
            (
                classify(header),
                seq_len,
            )
        )

    return records


rows = []

for r in download_manifest.itertuples(index=False):

    records = record_lengths_by_label(
        r.assembly_fasta_path
    )

    chromosome_bp = sum(
        length
        for label, length in records
        if label == "chromosome-labelled"
    )

    plasmid_bp = sum(
        length
        for label, length in records
        if label == "plasmid-labelled"
    )

    unlabelled_bp = sum(
        length
        for label, length in records
        if label == "unlabelled"
    )

    total_bp = (
        chromosome_bp
        + plasmid_bp
        + unlabelled_bp
    )

    rows.append(
        {
            "biosample": r.biosample,
            "assembly_accession": r.assembly_accession,
            "n_records": len(records),
            "chromosome_labelled_bp": chromosome_bp,
            "plasmid_labelled_bp": plasmid_bp,
            "unlabelled_bp": unlabelled_bp,
            "unlabelled_fraction": (
                unlabelled_bp / total_bp
                if total_bp > 0
                else float("nan")
            ),
            "direct_header_separation_complete": (
                chromosome_bp > 0
                and unlabelled_bp == 0
            ),
        }
    )

label_qc = pd.DataFrame(rows)
label_qc.to_csv(OUTPUT_PATH, index=False)

print("Direct chromosome/plasmid label assessment:")
print(
    "Assemblies with complete direct header separation:",
    int(label_qc["direct_header_separation_complete"].sum()),
    "/ 176",
)

print(
    "Assemblies containing plasmid-labelled sequence:",
    int((label_qc["plasmid_labelled_bp"] > 0).sum()),
)

print(
    "Assemblies containing unlabelled sequence:",
    int((label_qc["unlabelled_bp"] > 0).sum()),
)

print("\nUnlabelled sequence fraction:")
display(label_qc["unlabelled_fraction"].describe())

print("\nSaved:", OUTPUT_PATH)

print(
    "\nInterpretation: incomplete header labelling is not treated as biological "
    "absence or as automatic evidence that a contig is chromosomal. "
    "If direct separation is incomplete, a chromosome/plasmid classification "
    "step will be required before sequence-feature discovery."
)

print(
    "\nTransition: Cell 08.7 will quantify total sequence scale and "
    "summarize the feasibility criteria."
)

In [ ]:
#@title Cell 08.7 - Summarize sequence scale and feasibility criteria
# Combine mapping, download, sequence-QC, and chromosome-separation information.

import pandas as pd
from IPython.display import display

MANIFEST_PATH = (
    RESULTS_TABLE_DIR
    / "08_authoritative_176_assembly_manifest.csv"
)

DOWNLOAD_PATH = (
    RESULTS_TABLE_DIR
    / "08_downloaded_176_assembly_manifest.csv"
)

QC_PATH = (
    RESULTS_TABLE_DIR
    / "08_assembly_sequence_QC.csv"
)

LABEL_PATH = (
    RESULTS_TABLE_DIR
    / "08_chromosome_plasmid_label_QC.csv"
)

OUTPUT_PATH = (
    RESULTS_TABLE_DIR
    / "08_feasibility_summary.csv"
)

manifest = pd.read_csv(MANIFEST_PATH)
download = pd.read_csv(DOWNLOAD_PATH)
qc = pd.read_csv(QC_PATH)
label_qc = pd.read_csv(LABEL_PATH)

n_manifest = len(manifest)
n_downloaded = int(download["assembly_fasta_path"].notna().sum())
n_critical_qc = int(qc["critical_sequence_flag"].sum())
n_direct_separation = int(label_qc["direct_header_separation_complete"].sum())
n_need_classification = 176 - n_direct_separation

total_bp = int(qc["total_bp"].fillna(0).sum())
total_gbp = total_bp / 1e9

# Broad technical limit only; not a biological threshold.
scale_manageable = total_bp < 2_000_000_000

summary = pd.DataFrame(
    {
        "criterion": [
            "verified one-to-one assembly mappings",
            "downloaded readable genome FASTAs",
            "critical sequence-QC flags",
            "assemblies with complete direct header separation",
            "assemblies requiring chromosome/plasmid classification",
            "total assembled sequence Gbp",
            "sequence scale computationally manageable",
        ],
        "result": [
            n_manifest,
            n_downloaded,
            n_critical_qc,
            n_direct_separation,
            n_need_classification,
            total_gbp,
            scale_manageable,
        ],
    }
)

summary.to_csv(OUTPUT_PATH, index=False)

display(summary)

print("\nSaved:", OUTPUT_PATH)

print(
    "\nTransition: Cell 08.8 will issue the final GO/NO-GO decision "
    "for whole-chromosomal variable-sequence discovery."
)

In [ ]:
#@title Cell 08.8 - Final GO/NO-GO decision
# Decide whether the 176-genome whole-chromosomal comparison can proceed.

import pandas as pd

MANIFEST_PATH = (
    RESULTS_TABLE_DIR
    / "08_authoritative_176_assembly_manifest.csv"
)

DOWNLOAD_PATH = (
    RESULTS_TABLE_DIR
    / "08_downloaded_176_assembly_manifest.csv"
)

QC_PATH = (
    RESULTS_TABLE_DIR
    / "08_assembly_sequence_QC.csv"
)

LABEL_PATH = (
    RESULTS_TABLE_DIR
    / "08_chromosome_plasmid_label_QC.csv"
)

OUTPUT_PATH = (
    RESULTS_TABLE_DIR
    / "08_final_go_no_go_decision.csv"
)

manifest = pd.read_csv(MANIFEST_PATH)
download = pd.read_csv(DOWNLOAD_PATH)
qc = pd.read_csv(QC_PATH)
label_qc = pd.read_csv(LABEL_PATH)

mapping_ok = (
    len(manifest) == 176
    and manifest["biosample"].nunique() == 176
    and manifest["assembly_accession"].nunique() == 176
)

download_ok = (
    len(download) == 176
    and download["assembly_fasta_path"].notna().all()
)

sequence_qc_ok = (
    len(qc) == 176
    and int(qc["critical_sequence_flag"].sum()) == 0
)

total_bp = int(qc["total_bp"].fillna(0).sum())
scale_ok = total_bp < 2_000_000_000

n_need_classification = int(
    (~label_qc["direct_header_separation_complete"])
    .sum()
)

if not mapping_ok:
    decision = "NO-GO"
    reason = (
        "The 176-pathogen BioSample-to-assembly mapping is incomplete "
        "or not one-to-one."
    )

elif not download_ok:
    decision = "NO-GO"
    reason = (
        "Not all 176 exact assemblies were obtained as one readable genome FASTA."
    )

elif not sequence_qc_ok:
    decision = "NO-GO"
    reason = (
        "At least one assembly has a critical sequence-QC problem that must "
        "be reviewed before whole-sequence comparison."
    )

elif not scale_ok:
    decision = "NO-GO"
    reason = (
        "The total sequence scale exceeds the conservative feasibility limit."
    )

else:
    decision = "GO"

    if n_need_classification > 0:
        reason = (
            "The 176-genome comparison is technically feasible. "
            f"{n_need_classification} assemblies do not permit complete "
            "chromosome/plasmid separation from FASTA headers alone, so "
            "chromosome/plasmid contig classification must be completed "
            "before variable sequence elements are generated."
        )
    else:
        reason = (
            "The 176-genome comparison is technically feasible, and direct "
            "chromosome/plasmid separation is available for all assemblies."
        )

result = pd.DataFrame(
    {
        "item": [
            "mapping_OK",
            "download_OK",
            "sequence_QC_OK",
            "sequence_scale_OK",
            "assemblies_requiring_chromosome_plasmid_classification",
            "final_decision",
            "reason",
        ],
        "result": [
            mapping_ok,
            download_ok,
            sequence_qc_ok,
            scale_ok,
            n_need_classification,
            decision,
            reason,
        ],
    }
)

result.to_csv(OUTPUT_PATH, index=False)

print("Final decision:", decision)
print(reason)

print("\nSaved:", OUTPUT_PATH)

print(
    "\nNotebook 08 ends here. "
    "Do not generate variable sequence elements until this decision has "
    "been reviewed."
)